# Experiment 7: Text Preprocessing and Embedding Generation Using NLP Libraries



**Dataset:** Brown Corpus — `news` category  
**Tools:** Python, NLTK, Gensim, NumPy, Google Colab


In [1]:
# Student details
name = "Midhun P"
roll_number = "24BAD069"

print("Name:", name)
print("Roll Number:", roll_number)

Name: Midhun P
Roll Number: 24BAD069


## Setup: Install libraries and download NLTK resources

In [2]:
%pip -q install nltk gensim numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 7.0 MB/s eta 0:00:00


In [3]:
import re
import nltk
import numpy as np
from nltk.corpus import brown, stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from gensim.models import Word2Vec

nltk.download("brown", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("Libraries imported and NLTK resources downloaded successfully.")

Libraries imported and NLTK resources downloaded successfully.


## Part A: Dataset Preparation and Text Preprocessing

Load sample sentences from the Brown Corpus's `news` category, convert text to lowercase, tokenize it, remove punctuation and stop words, and lemmatize the remaining words.

In [4]:
category = "news"
raw_sentences = brown.sents(categories=category)

# Keep non-empty sentences and convert tokens to lowercase for display/processing.
original_sentences = [
    " ".join(sentence) for sentence in raw_sentences if sentence
]

print("Selected Brown Corpus category:", category)
print("\nSample original sentences:")
for sentence in original_sentences[:5]:
    print("-", sentence)

Selected Brown Corpus category: news

Sample original sentences:
- The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .
- The jury further said in term-end presentments that the City Executive Committee , which had over-all charge of the election , `` deserves the praise and thanks of the City of Atlanta '' for the manner in which the election was conducted .
- The September-October term jury had been charged by Fulton Superior Court Judge Durwood Pye to investigate reports of possible `` irregularities '' in the hard-fought primary which was won by Mayor-nominate Ivan Allen Jr. .
- `` Only a relative handful of such reports was received '' , the jury said , `` considering the widespread interest in the election , the number of voters and the size of this city '' .
- The jury said it did find that many of Georgia's registration and election laws `` are outmoded or inadequate and of

In [5]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_sentence(sentence):
    # Lowercase and tokenize.
    tokens = word_tokenize(sentence.lower())
    # Keep alphabetic words only: this removes punctuation and numbers.
    tokens = [token for token in tokens if token.isalpha()]
    # Remove stop words and lemmatize.
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]
    return tokens

# Process the corpus sentence by sentence.
processed_sentences = [
    preprocess_sentence(sentence)
    for sentence in original_sentences
]
processed_sentences = [tokens for tokens in processed_sentences if tokens]

print("Original and preprocessed text:")
for original, cleaned in zip(original_sentences[:5], processed_sentences[:5]):
    print("\nOriginal:     ", original)
    print("Preprocessed: ", " ".join(cleaned))

Original and preprocessed text:

Original:      The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .
Preprocessed:  fulton county grand jury said friday investigation atlanta recent primary election produced evidence irregularity took place

Original:      The jury further said in term-end presentments that the City Executive Committee , which had over-all charge of the election , `` deserves the praise and thanks of the City of Atlanta '' for the manner in which the election was conducted .
Preprocessed:  jury said presentment city executive committee charge election deserves praise thanks city atlanta manner election conducted

Original:      The September-October term jury had been charged by Fulton Superior Court Judge Durwood Pye to investigate reports of possible `` irregularities '' in the hard-fought primary which was won by Mayor-nominate Ivan Allen Jr. .
Preprocessed:  te

## Part B: Vocabulary Generation and Word Frequency Analysis

Create the vocabulary from preprocessed tokens, count frequencies using `FreqDist`, and print the ten most common words.

In [6]:
all_tokens = [word for sentence in processed_sentences for word in sentence]
vocabulary = sorted(set(all_tokens))
frequency_distribution = FreqDist(all_tokens)

print("Vocabulary size:", len(vocabulary))
print("Total preprocessed tokens:", len(all_tokens))
print("\nTen most frequent words:")
for word, frequency in frequency_distribution.most_common(10):
    print(f"{word}: {frequency}")

print("\nFirst 20 vocabulary words:")
print(vocabulary[:20])
print("\nNumber of tokenized sentences prepared for Word2Vec:", len(processed_sentences))

Vocabulary size: 9896
Total preprocessed tokens: 47682

Ten most frequent words:
said: 406
year: 260
would: 249
new: 241
state: 223
one: 222
last: 177
two: 174
first: 158
president: 155

First 20 vocabulary words:
['aaron', 'abandoned', 'abandonment', 'abatuno', 'abbey', 'abe', 'abel', 'abell', 'abide', 'ability', 'ablaze', 'able', 'abner', 'aboard', 'abolish', 'abolition', 'abortive', 'abra', 'abraham', 'abroad']

Number of tokenized sentences prepared for Word2Vec: 4590


## Part C: Word Embedding Generation Using Word2Vec

Train Word2Vec with Skip-gram (`sg=1`), 100-dimensional vectors, and a context window of 5. The `min_count=2` setting ignores words that occur only once.

In [7]:
model = Word2Vec(
    sentences=processed_sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # Skip-gram architecture
    epochs=20,
    seed=42
)

print("Word2Vec model training completed.")
print("Vocabulary learned by Word2Vec:", len(model.wv))

Word2Vec model training completed.
Vocabulary learned by Word2Vec: 5164


In [8]:
# Choose a common word that exists in the trained model.
preferred_words = ["government", "state", "people", "country", "world", "president"]
selected_word = next((word for word in preferred_words if word in model.wv), None)

if selected_word is None:
    selected_word = next(iter(model.wv.index_to_key))

embedding_vector = model.wv[selected_word]
print("Selected word:", selected_word)
print("Embedding vector:")
print(embedding_vector)
print("Vector dimensions:", embedding_vector.shape)
print("Number of dimensions:", len(embedding_vector))

print("\nFive most similar words:")
similar_words = model.wv.most_similar(selected_word, topn=5)
for word, score in similar_words:
    print(f"{word}: {score:.4f}")

Selected word: government
Embedding vector:
[-0.0023958   0.06620263  0.18245114  0.26080492  0.11600708 -0.12291774
  0.28581658  0.47908637  0.15904164 -0.7470618  -0.539586   -0.13982673
  0.08052105  0.5070871  -0.2148737  -0.5231489   0.10316352 -0.4225306
 -0.62491685  0.2458907   0.2582314   0.28439635 -0.40163162  0.10257223
 -0.1782406   0.11809082 -0.2846347   0.19338495 -0.3654937  -0.19342208
 -0.25121227 -0.19694257 -0.16773745 -0.11358507  0.07961982 -0.6840771
  0.2665761  -0.04812219 -0.3546116  -0.21022762 -0.07665117 -0.32203984
  0.15130596 -0.5597363   0.42138913  0.0488498   0.4016701   0.5887211
  0.7358711  -0.42180622  0.16313055 -0.17341349 -0.06182995  0.2133664
  0.08838648 -0.04332093 -0.07508589 -0.03324312  0.2653698  -0.05852769
  0.13957812 -0.39886758 -0.20226626 -0.22087769  0.21463367 -0.16446899
 -0.20336314 -0.00954103 -0.70323765 -0.3346655   0.07445148  0.03652648
  0.0926655  -0.5230936   0.15529466  0.10821547  0.19272503 -0.5687058
 -0.31861526

## Part D: Word Similarity Analysis and Experiment Management

Select three word pairs that exist in the trained vocabulary. Calculate cosine similarity using Word2Vec and compare the scores. The notebook checks vocabulary membership so it does not fail if a suggested word is absent.

In [9]:
candidate_pairs = [
    ("government", "state"),
    ("people", "man"),
    ("country", "world"),
    ("president", "government"),
    ("house", "city"),
    ("school", "college"),
]

valid_pairs = []
for word_a, word_b in candidate_pairs:
    if word_a in model.wv and word_b in model.wv:
        valid_pairs.append((word_a, word_b))
    if len(valid_pairs) == 3:
        break

# If fewer than three candidates are present, choose additional words from the learned vocabulary.
if len(valid_pairs) < 3:
    keys = model.wv.index_to_key
    for i in range(min(len(keys), 30)):
        for j in range(i + 1, min(len(keys), 30)):
            pair = (keys[i], keys[j])
            if pair not in valid_pairs:
                valid_pairs.append(pair)
            if len(valid_pairs) == 3:
                break
        if len(valid_pairs) == 3:
            break

similarity_results = []
for word_a, word_b in valid_pairs:
    score = float(model.wv.similarity(word_a, word_b))
    similarity_results.append((word_a, word_b, score))
    print(f"Cosine similarity ({word_a}, {word_b}) = {score:.4f}")

if similarity_results:
    most_similar_pair = max(similarity_results, key=lambda item: item[2])
    print("\nMost similar pair:", most_similar_pair[0], "-", most_similar_pair[1])
    print("Highest cosine similarity:", f"{most_similar_pair[2]:.4f}")
    print("Interpretation: a higher score indicates that the two words have more similar learned vector directions.")
else:
    print("Not enough words were available to compare pairs.")

Cosine similarity (government, state) = 0.4911
Cosine similarity (people, man) = 0.5994
Cosine similarity (country, world) = 0.4605

Most similar pair: people - man
Highest cosine similarity: 0.5994
Interpretation: a higher score indicates that the two words have more similar learned vector directions.


## Observation and Conclusion

Text preprocessing converted the Brown Corpus text into normalized tokens by lowercasing, removing punctuation and stop words, and lemmatizing. `FreqDist` showed the most common terms and the vocabulary size. Word2Vec generated a 100-dimensional vector for each eligible word and returned words with similar contexts. Cosine similarity compared the vector directions of selected word pairs; higher scores indicate greater similarity in the learned embedding space. Results can vary slightly with corpus content and training settings.